In [6]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(override=True)

DATA_PATH = os.getenv("DATA_PATH")

if DATA_PATH is None:
    raise ValueError("DATA_PATH is not defined in .env")

data_dir = Path(DATA_PATH)

# Load base dataset
df_combined = pd.read_csv(data_dir / "df_n1.csv")

print("Original shape:", df_combined.shape)

# Find all n2 generated files
files = list(data_dir.glob("*n2*.csv"))

print(f"Found {len(files)} n2 files:")
for f in files:
    print(f" - {f.name}")


for file in files:

    try:
        df_n2 = pd.read_csv(file)

        if df_n2.empty:
            print(f"[SKIPPED EMPTY] {file.name}")
            continue


        df_combined = df_combined.merge(
            df_n2,
            on="patch_id",
            how="left",
            suffixes=("", f"_{file.stem}")
        )

    except pd.errors.EmptyDataError:
        print(f"[SKIPPED NO DATA] {file.name}")
        continue

    except Exception as e:
        print(f"[SKIPPED ERROR] {file.name}: {e}")
        continue


df_combined = df_combined.sort_values(
    by="patch_id"
)
print("Final shape:", df_combined.shape)

Original shape: (10, 19)
Found 2 n2 files:
 - df_n2_1.csv
 - df_n2_2.csv
[MERGING] df_n2_1.csv -> (10, 2)
[MERGING] df_n2_2.csv -> (10, 3)
Final shape: (10, 22)


In [3]:
df_combined.to_csv(f'{DATA_PATH}df_n2.csv', index=False)